The overall flow for the project is going to be:


* Process all textfiles:
* * search the file 
* * score the file
* * create the biblio info
* * dump the csv score sheets
* * Create the JSON files
* Generate comparison data:
* * Create Lit Level scores
* * Create Comparison sheets
* * Dump Comparisions to csvs
* Create reverse text location to publication list csv for each location
* create tableau csvs

In [ ]:
#imports external tools
import pandas as pd
import roman
import os
import re
import json
import csv
#import project tools
from functions_and_classes.paper_processing import process_and_score_paper as psp
#psp takes a filename, dataframe as input. The data frame should be a dataframe that links file names to bibliographic data.
from data.biblio_data import dl_files
#dl_files is a data frame with bibliographic information
from data.txts import txt_list
#txt_list is a list of all text files to be processed, use it with psp and dl_files to process all existing txt files
from functions_and_classes.classes import *
#gets the paper objects
from functions_and_classes.paper_io import *
#gets the paper i/o functions p_from_dict, etc.

In [ ]:
#the psp function with dl_files will scan, score, biblio update. I need to then dump the csv scores and create the jsons
#one thing to note is that sometimes the json writes are buggy and one or two won't load. just recreate them manually again
for file in txt_list:
    #i can process the papers using the text file name because the Paper init 
    #will create a name that chops off the .txt
    paper = psp(file)
    paper.csvScoreDump()
    jf = open('data/jsons/'+paper.name+'.json', 'w')
    jd = p_to_dict(paper)
    json.dump(jd,jf)
    jf.close
    print(file, ' processed, csvs dumped, json created')

In [ ]:
#To generate the literature level data csvs, I can use the function lit_scores_from_jsons() from functions_and_classes.lit_level_operations
from functions_and_classes.lit_level_operations import lit_scores_from_jsons, lit_to_csv

In [ ]:
lit_scores = lit_scores_from_jsons()
#creates a dictionary of score dictionaries
lit_to_csv(lit_scores)
#dumps the csvs

In [ ]:
from functions_and_classes.comparisons import updateCompData
#now we can update the comp data, though it might be better to handle this one function at a time
#..sometimes it hangs for a while and seems to quit

In [ ]:
updateCompData()
#updates the comp data csvs

In [ ]:
from data.comparison_data import a_l_c_comp, s_l_c_comp, a_l_p_comp, s_l_p_comp

In [ ]:
print(a_l_p_comp['totalAggressiveCites']['Literature'])
print(s_l_p_comp['totalStrictCites']['Literature'])

In [ ]:
#now we need to put together the reverse lookup data
#For imports, we first need to get a list of treatise paragraphs:
from functions_and_classes.treatise_reference_data import treatise_paragraph_list
#to turn that into a list of chapters, we'll need the chapter scores function:
from functions_and_classes.paper_processing import chapterScores

In [ ]:
#first create all the dictionaries to use to create the data frames:

#start with paragraphs, we'll do chapter after
a_w_p_reverse = {}
s_w_p_reverse = {}

#set every paragraph equal to zero (there should be 1375 paragraphs)
for para in treatise_paragraph_list:
    a_w_p_reverse[para] = 0
    s_w_p_reverse[para] = 0
print(len(a_w_p_reverse.keys()))
print(len(s_w_p_reverse.keys()))
#now we'll turn that into a chapter score card with each chapter set to zero (there should be 93 chapters)
a_w_c_reverse = chapterScores(a_w_p_reverse)
s_w_c_reverse = chapterScores(s_w_p_reverse)
print(len(a_w_c_reverse.keys()))
print(len(s_w_c_reverse.keys()))

In [ ]:
#now we need to put together the reverse generations

#generate the blank data frames based on the dictionaries above
awprevdf = pd.DataFrame.from_dict(a_w_p_reverse, orient='index', columns=['Blank'])
awcrevdf = pd.DataFrame.from_dict(a_w_c_reverse, orient='index', columns=['Blank'])
swprevdf = pd.DataFrame.from_dict(s_w_p_reverse, orient='index', columns=['Blank'])
swcrevdf = pd.DataFrame.from_dict(s_w_c_reverse, orient='index', columns=['Blank'])

#fill them with the article data
for file in json_list:
    #create the paper object
    jf = open('data/jsons/'+file, 'r')
    jd = json.load(jf)
    jf.close()
    pawp = pd.DataFrame.from_dict(jd['a_w_p'], orient='index', columns=[file[:-5]])
    awprevdf[file[:-5]] = pawp[file[:-5]]
    pawc = pd.DataFrame.from_dict(jd['a_w_c'], orient='index', columns=[file[:-5]])
    awcrevdf[file[:-5]] = pawc[file[:-5]]
    pswp = pd.DataFrame.from_dict(jd['s_w_p'], orient='index', columns=[file[:-5]])
    swprevdf[file[:-5]] = pswp[file[:-5]]
    pswc = pd.DataFrame.from_dict(jd['s_w_c'], orient='index', columns=[file[:-5]])
    swcrevdf[file[:-5]] = pswc[file[:-5]]

#delete the blanks
del awprevdf['Blank']
del awcrevdf['Blank']
del swprevdf['Blank']
del swcrevdf['Blank']

#generate csvs
awprevdf.to_csv('data/awpreverse.csv')
awcrevdf.to_csv('data/awcreverse.csv')
swprevdf.to_csv('data/swpreverse.csv')
swcrevdf.to_csv('data/swcreverse.csv')

In [ ]:
#let's reimport now the csvs to make sure they're good to go
from data.reverse_loc_to_pubs import awprev_df, awcrev_df, swprev_df, swcrev_df

In [ ]:
#finally, let's test them. to do that we need to import a function from lit_level_operations:
from functions_and_classes.lit_level_operations import locationFrequency

In [ ]:
#now let's test:
print('awprev')
for pair in locationFrequency('1.4.6.5', awprev_df)[:3]:
    print(pair[1],pair[2])
print('swprev')
for pair in locationFrequency('1.4.6.5', swprev_df)[:3]:
    print(pair[1],pair[2])
print('awcrev')
for pair in locationFrequency('1.4.6', awcrev_df)[:3]:
    print(pair[1],pair[2])
print('swcrev')
for pair in locationFrequency('1.4.6', swcrev_df)[:3]:
    print(pair[1],pair[2])

In [ ]:
#Next, we want to update the tableau CSVs

In [ ]:
header_row = ["Location", "Score", "Author", "Title", "Year"]
agg_para = open("data/agg_para_tableau.csv", 'w')
ap_csv = csv.writer(agg_para)
ap_csv.writerow(header_row)
agg_chap = open("data/agg_chap_tableau.csv", 'w')
ac_csv = csv.writer(agg_chap)
ac_csv.writerow(header_row)
strict_chap = open("data/strict_chap_tableau.csv", 'w')
sc_csv = csv.writer(strict_chap)
sc_csv.writerow(header_row)
strict_para = open("data/strict_para_tableau.csv", 'w')
sp_csv = csv.writer(strict_para)
sp_csv.writerow(header_row)

for file in json_list:
    filepath = 'data/jsons/'+file
    jf = open(filepath,'r')
    jd = json.load(jf)
    jf.close()
    paper = p_from_dict(jd)
 

    if paper.totalStrictCites > 0:
        for para in paper.s_w_p:
            if paper.s_w_p[para] > 0:
                sp_row = []
                sp_row.append(para)
                sp_row.append(round(paper.s_w_p[para],3))
                sp_row.append(paper.biblio['Author'])
                sp_row.append(paper.biblio['Title of Work'])
                sp_row.append(int(paper.biblio['Year']))
                sp_csv.writerow(sp_row)
        for chap in paper.s_w_c:
            if paper.s_w_c[chap] > 0:
                sc_row = []
                sc_row.append(chap)
                sc_row.append(round(paper.s_w_c[chap],3))
                sc_row.append(paper.biblio['Author'])
                sc_row.append(paper.biblio['Title of Work'])
                sc_row.append(int(paper.biblio['Year']))
                sc_csv.writerow(sc_row)
        
    if paper.totalAggressiveCites > 0:
        for para in paper.a_w_p:
            if paper.a_w_p[para] > 0:
                ap_row = []
                ap_row.append(para)
                ap_row.append(round(paper.a_w_p[para],3))
                ap_row.append(paper.biblio['Author'])
                ap_row.append(paper.biblio['Title of Work'])
                ap_row.append(int(paper.biblio['Year']))
                ap_csv.writerow(ap_row)
                
        for chap in paper.a_w_c:
            if paper.a_w_c[chap] > 0:
                ac_row = []
                ac_row.append(chap)
                ac_row.append(round(paper.a_w_c[chap],3))
                ac_row.append(paper.biblio['Author'])
                ac_row.append(paper.biblio['Title of Work'])
                ac_row.append(int(paper.biblio['Year']))
                ac_csv.writerow(ac_row)

agg_para.close()
agg_chap.close()
strict_chap.close()
strict_para.close()
print('done generating all the csvs for tableau!')

In [ ]:
#From here everything is ready to go. Just need to reopen Tableau to refresh.
#The next step to add is just a single-article at a time add. Probably just use all the lit and comparative stuff would 
#need to be redone wholly, but the single psp, csv dump, and json dump could be wrapped up here:

In [ ]:
new_pub_to_add = ""
paper_to_add = psp(new_pub_to_add)
paper.csvScoreDump()
new_jf = open('data/jsons/'+paper.name+'.json', 'w')
new_jd = p_to_dict(paper)
json.dump(new_jd,new_jf)
jf.close
print(paper.name, ' processed, csvs dumped, json created')

Now I'm going to swap develop the gcloud storage interactions.
The list of static files to be put in the storage cloud seems to be:
* lit files; e.g. l_a_w_p etc.
* comparison files; e.g. a_l_p_comp etc.
* reverse lookup files; e.g. awpreverse.csv etc.
* I'm going to store the score csvs in the firebase data slots so those csvs don't need to be stored here
* two treatise-reference-data sources
* full text in csv
* all the txt files that are searched by the paper object that's created
* The treatise chapter and paragraph stuff along with the tableau csvs can stay local since they're connected here to the tableau program

In [ ]:
from google.cloud import firestore
from google.cloud import storage

In [ ]:
import os

In [ ]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS']='./treatise-mapping-project-f547afe7f49d.json'

In [ ]:
os.environ['GCLOUD_PROJECT']='treatise-mapping-project'

In [ ]:
stor_client = storage.Client()

In [ ]:
base_bucket = stor_client.bucket('treatise-mapping-project.appspot.com')

The basic way to use Storage as a file storage is as follows:
* First, create a blob: base_bucket.blob('Filename')
* Second, download that blob to string: blob.download_as_string()
* Third, use io.BytesIO to create the equivalent of a file object: fileObject = io.BytesIO(blob.download_as_string())
* If the file is a CSV you can import it into a dataframe: pd.read_csv(fileObject) (+ if it's one of my csvs for comps or whatever, you can also specify index_col=0 at this stage)
* If the file is a json you can create the dictionary by json.load(fileObject)
* By default fileObject will just be treated as equivalent of a txt file with lines etc.

In [ ]:
import io

In [ ]:
#this function takes care of pulling a csv and generating the data frame.
#It's what I need. I'll create a function sheet for cloud storage io
def df_from_gc_csv(csv_name):
    sc = storage.Client()
    bkt = sc.bucket('treatise-mapping-project.appspot.com')
    blob = bkt.blob(csv_name)
    dl_data = io.BytesIO(blob.download_as_string())
    df = pd.read_csv(dl_data, index_col=0)
    
    return df

In [ ]:
##This does the same thing but for JSON objects
def dic_from_gc_json(json_name):
    sc = storage.Client()
    bkt = sc.bucket('treatise-mapping-project.appspot.com')
    blob = bkt.blob(json_name)
    dl_data = io.BytesIO(blob.download_as_string())
    do = json.load(dl_data)

    return do

In [ ]:
#Same thing but for txt files, returns a list of lines
def txt_from_gc_txt(txt_name):
    sc = storage.Client()
    bkt = sc.bucket('treatise-mapping-project.appspot.com')
    blob = bkt.blob(txt_name)
    dl_data = blob.download_as_string()
    usable_string = dl_data.decode('utf-8')
    lines_split = usable_string.split('\n')
    
    return lines_split

In [ ]:
#Here's a function to upload to a given file name in the bucket from a given path
def upload_to_gc(file_name, file_path):
    sc = storage.Client()
    bkt = sc.bucket('treatise-mapping-project.appspot.com')
    blob = bkt.blob(file_name)
    blob.upload_from_filename(file_path)
    
    print(file_name, 'uploaded to gcloud storage')
    

To use the download functions within the code,
* create a functions_and_classes.cloud_io module
* put these upload and download functions
* anywhere in the current code i find an import of a csv, replace it with the from gc csv function on the file name.
* anywhere in the current code i find an import of a json, replace it with the gc json function on the same name
* anywhere in the current code i find an import of a txt file, replace it with the gc txt functino on the same name.

The upload functino I've described above works with files and takes a file path and a file name. IT's good for the one time upload of files. What I want to do below is to come up with a function that will write from a file object or data directly to the storage bucket.

In [ ]:
upload_to_gc('strict_para_tableau.csv', 'data/strict_para_tableau.csv')
upload_to_gc('strict_chap_tableau.csv', 'data/strict_chap_tableau.csv')
upload_to_gc('agg_para_tableau.csv', 'data/agg_para_tableau.csv')
upload_to_gc('agg_chap_tableau.csv', 'data/agg_chap_tableau.csv')
print('Tableau CSVs uploaded')

In [ ]:
upload_to_gc('swcreverse.csv', 'data/swcreverse.csv')
upload_to_gc('swpreverse.csv', 'data/swpreverse.csv')
upload_to_gc('awcreverse.csv', 'data/awcreverse.csv')
upload_to_gc('awpreverse.csv', 'data/awpreverse.csv')
print('Reverse Lookup CSVS uploaded')

In [ ]:
upload_to_gc('s_l_p_comp.csv', 'data/s_l_p_comp.csv')
upload_to_gc('s_l_c_comp.csv', 'data/s_l_c_comp.csv')
upload_to_gc('a_l_p_comp.csv', 'data/a_l_p_comp.csv')
upload_to_gc('a_l_c_comp.csv', 'data/a_l_c_comp.csv')
print('Comp CSVS uploaded')

In [ ]:
upload_to_gc('lit_a_l_c.csv', 'data/csvs/lit_a_w_c.csv')
upload_to_gc('lit_a_l_p.csv', 'data/csvs/lit_a_w_p.csv')
upload_to_gc('lit_s_l_c.csv', 'data/csvs/lit_s_w_c.csv')
upload_to_gc('lit_s_l_p.csv', 'data/csvs/lit_s_w_p.csv')
upload_to_gc('lit_a_w_c.csv', 'data/csvs/lit_a_w_c.csv')
upload_to_gc('lit_a_w_p.csv', 'data/csvs/lit_a_w_p.csv')
upload_to_gc('lit_s_w_c.csv', 'data/csvs/lit_s_w_c.csv')
upload_to_gc('lit_s_w_p.csv', 'data/csvs/lit_s_w_p.csv')
print('Lit Level Data Uploaded')

In [ ]:
upload_to_gc('Norton to SBN Dictionary.txt', 'functions_and_classes/Norton to SBN Dictionary.txt')
upload_to_gc('SBN to Norton Dictionary.txt', 'functions_and_classes/SBN to Norton Dictionary.txt')
print('Treatise Reference materials txts uploaded')



In [ ]:
def upload_csv_from_df(file_name, df):
    #THIS SEEMS TO NOT WORK AS WELL AS THE df_from_gc_csv with file and file_path works!
    sc = storage.Client()
    bkt = sc.bucket('treatise-mapping-project.appspot.com')
    blob = bkt.blob(file_name)
    file_obj = io.StringIO(df.to_csv())
    blob.upload_from_file(file_obj)
    print(file_name, 'uploaded!')

In [ ]:
refs = pd.read_csv('data/List-of-Works.csv', header=0)

In [ ]:
upload_to_gc('List-of-Works.csv', 'data/List-of-Works.csv')

In [ ]:
df_from_gc_csv('List-of-Works.csv')

In [ ]:
def upload_json_from_dict(file_name, dic_in):
    #enter a name for the file and the dictinoary, will upload json.
    sc = storage.Client()
    bkt = sc.bucket('treatise-mapping-project.appspot.com')
    blob = bkt.blob(file_name)
    file_obj = io.StringIO(json.dumps(dic_in))
    blob.upload_from_file(file_obj)
    print(file_name, 'uploaded!')

OK, so the functions above allow for uploading straight from a file into the cloud. I can replace writes to csv and json dump in the project management with these functions.

I don't at the moment need to be writing any text files so I can hold off on that.

# Now we'll start working with the Firestore clients to create the paper objects in Firestore

In [ ]:
firedb = firestore.Client()

In [ ]:
from data.jsons import json_list
from functions_and_classes.paper_io import *

In [ ]:
def upload_paper_dict_to_firedb(paper_dict):
    base_line = {}
    base_line['biblio'] = paper_dict['biblio']
    base_line['name'] = paper_dict['name']
    base_line['totalAggressiveCites'] = paper_dict['totalAggressiveCites']
    base_line['totalStrictCites'] = paper_dict['totalStrictCites']
    base_line['rawNortonScore'] = paper_dict['rawNortonScore']
    base_line['rawSbnScore'] = paper_dict['rawSbnScore']
    base_line['rawTScore'] = paper_dict['rawTScore']
    base_line['rawPageNumScore'] = paper_dict['rawPageNumScore']

    #set the basic information
    basic_ref = firedb.collection(u'publications').document(paper_dict['name'])
    basic_ref.set(base_line)

    #set the cites
    cites_ref = basic_ref.collection(u'citations')
    cites = {}
    cites[('nortonCites')] = paper_dict['nortonCites']
    cites[('sbnCites')] = paper_dict['sbnCites']
    cites[('tCites')] = paper_dict['tCites']
    cites[('pageNumCites')] = paper_dict['pageNumCites']
    cites[('rawParenthesesCapture')] = paper_dict['rawParenthesesCapture']
    cites_ref.document('cites').set(cites)


    #set the scores
    scores_ref = basic_ref.collection(u'scores')
    scores_ref.document('a_l_p').set(paper_dict['a_l_p'])
    scores_ref.document('a_l_c').set(paper_dict['a_l_c'])
    scores_ref.document('a_w_p').set(paper_dict['a_w_p'])
    scores_ref.document('a_w_c').set(paper_dict['a_w_c'])
    scores_ref.document('s_l_p').set(paper_dict['s_l_p'])
    scores_ref.document('s_l_c').set(paper_dict['s_l_c'])
    scores_ref.document('s_w_p').set(paper_dict['s_w_p'])
    scores_ref.document('s_w_c').set(paper_dict['s_w_c'])
    
    print('done uploading', paper_dict['name'])

In [ ]:
#batch upload for initial load
for fil in json_list:
    jfiler = open('data/jsons/'+fil, 'r')
    jd = json.load(jfiler)
    jfiler.close()
    upload_paper_dict_to_firedb(jd)

print('done uploading files')

In [ ]:
#now to work on a p_dict_from_fire
def p_dict_from_fire(fname):
    doc_ref = firedb.collection('publications').document(fname)
    #get and set basic info
    base_dict = doc_ref.get().to_dict()
    od = {}
    od['biblio'] = base_dict['biblio']
    od['name'] = base_dict['name']
    od['totalAggressiveCites'] = base_dict['totalAggressiveCites']
    od['totalStrictCites'] = base_dict['totalStrictCites']
    od['rawNortonScore'] = base_dict['rawNortonScore']
    od['rawSbnScore'] = base_dict['rawSbnScore']
    od['rawTScore'] = base_dict['rawTScore']
    od['rawPageNumScore'] = base_dict['rawPageNumScore']
    #get and set cites
    cites_ref = doc_ref.collection('citations').document('cites')
    cites_dict = cites_ref.get().to_dict()
    od['nortonCites'] = cites_dict['nortonCites']
    od['sbnCites'] = cites_dict['sbnCites']
    od['tCites'] = cites_dict['tCites']
    od['pageNumCites'] = cites_dict['pageNumCites']
    od['rawParenthesesCapture'] = cites_dict['rawParenthesesCapture']
    #get and set scores
    scores_ref = doc_ref.collection('scores')
    od['a_l_p'] = scores_ref.document('a_l_p').get().to_dict()
    od['a_l_c'] = scores_ref.document('a_l_c').get().to_dict()
    od['a_w_p'] = scores_ref.document('a_w_p').get().to_dict()
    od['a_w_c'] = scores_ref.document('a_w_c').get().to_dict()
    od['s_l_p'] = scores_ref.document('s_l_p').get().to_dict()
    od['s_l_c'] = scores_ref.document('s_l_c').get().to_dict()
    od['s_w_p'] = scores_ref.document('s_w_p').get().to_dict()
    od['s_w_c'] = scores_ref.document('s_w_c').get().to_dict()
    print(fname, ' recovered into a paper dict ready form')
    return od

In [ ]:
def paper_from_fire(filename):
    d_from_fire = p_dict_from_fire(filename)
    paper = p_from_dict(d_from_fire)
    return paper

In [ ]:
def paper_to_fire(paper_obj):
    p_dict = p_to_dict(paper_obj)
    upload_paper_dict_to_firedb(p_dict)
    print('Done uploading', paper_obj.name)

# All the necessary paper-io and database interactions are done. the only thing i'll need to do now is to replace the local file functionality with cloud file functionality!

In [ ]:
def updateFileToTitleDict():
    #just run this function to update alongside the other updates to static files
    file_to_title_dict = {}
    for pub in firedb.collection('publications').stream():
        basedict = firedb.collection('publications').document(pub.id).get().to_dict()
        file_to_title_dict[basedict['name']] = basedict['biblio']['Title of Work']
    upload_json_from_dict('file_to_title_dict.json', file_to_title_dict)
    

In [ ]:
#testing some of the lit gen functions for comp data
s_l_pd = df_from_gc_csv('lit_s_l_p.csv').rename(columns={'Score':'Literature'})

In [ ]:
s_l_pd

In [ ]:
l_a_l_p = df_from_gc_csv('lit_a_l_p.csv')


In [ ]:
s_l_pdd = l_a_l_p.rename(columns={'Score':'Literature'})
s_l_pdd

In [ ]:
#testing the lit comp generations:
ftn = 'PENTIH'
file_p_dict = p_dict_from_fire(ftn)

In [ ]:
test_lit = df_from_gc_csv('lit_s_w_c.csv')

In [ ]:
test_dict_dict = test_lit.to_dict()

In [ ]:
test_dict_dict['Score']['1.1.2']

In [ ]:
chapters = test_dict_dict['Score'].keys()

In [ ]:
redone_lit = {}
for chap in chapters:
    redone_lit[chap] = test_dict_dict['Score'][chap]

In [ ]:
redone_lit['1.1.2']

In [ ]:
from io import StringIO
import csv

In [ ]:
ftester = StringIO()
writerc = csv.writer(ftester)
writerc.writerow(('Chapter', 'Score'))
for chap in redone_lit.keys():
    chapter = chap
    score = redone_lit[chap]
    writerc.writerow((chapter, score))

In [ ]:
from io import BytesIO

In [ ]:
def write_lit_chap_dict_to_gc_csv(file_name, din):
    sio = StringIO()
    wtr = csv.writer(sio)
    wtr.writerow(('Chapter', 'Score'))
    for chap in din.keys():
        wtr.writerow((chap, din[chap]))
    sc = storage.Client()
    bkt = sc.bucket('treatise-mapping-project.appspot.com')
    blob = bkt.blob(file_name+'.csv')
    blob.upload_from_string(sio.getvalue())
    sio.close()
    print('done uploading', file_name)
        
def write_lit_para_dict_to_gc_csv(file_name, din):
    sio = StringIO()
    wtr = csv.writer(sio)
    wtr.writerow(('Paragraph', 'Score'))
    for para in din.keys():
        wtr.writerow((para, din[para]))
    sc = storage.Client()
    bkt = sc.bucket('treatise-mapping-project.appspot.com')
    blob = bkt.blob(file_name+'.csv')
    blob.upload_from_string(sio.getvalue())
    sio.close()
    print('done uploading', file_name)
        
        

In [ ]:
write_lit_para_dict_to_gc_csv('test-lit-sheet4', redone_lit)

In [ ]:
redl_df = df_from_gc_csv('test-lit-sheet4.csv').rename(columns={'Score':'Literature'})
redl_df


In [ ]:
firedb = firestore.Client()

In [ ]:
baseref = firedb.collection('publications').document('YANHVO').collection('scores')

In [ ]:
docref = baseref.document('a_w_c')

In [ ]:
testfired = docref.get().to_dict()
len(testfired.keys())

In [ ]:
testfiredf = pd.DataFrame(index=range(0,len(testfired.keys())) ,columns=['Location', 'Score'])

In [ ]:
testfiredf['Location'] = testfired.keys()

In [ ]:
testfiredf['Score'] = testfired.values()

In [ ]:
firedb.collection('publications').document('YANHVO').get().to_dict()['totalAggressiveCites']#not sure this is working

In [ ]:
testfiredf.sort_values('Score', ascending=False)

In [ ]:
testfired['1.4.6']

In [ ]:
dlistpairs = []
for pair in doc_dict.items():
    key = pair[0]
    value = pair[1]
    lp = [key, value]
    dlistpairs.append(lp)

In [ ]:
redone_df = pd.DataFrame(dlistpairs, columns=['Location','Score'])

In [ ]:
redone_df.loc[redone_df['Location']=='1.4.6']

In [ ]:
nz = df['Count'] > 0
    scdf = df[nz].sort_values('Count', ascending=False)
    entries = scdf['Location'].count()
    out_list = []
    for i in range(0,entries):
        pair = (scdf.iloc[i]['Location'], round(scdf.iloc[i]['Count'],3))
        out_list.append(pair)
    return out_list

In [ ]:
snd = txt_from_gc_txt("SBN to Norton Dictionary.txt")

In [ ]:
tpl = []
mss = {}
ntsd = {}

In [ ]:
for line in nsd[:-1]:
    para = line.split(" : ")[0].strip()
    tpl.append(para)
    pages = line.split(" : ")[1].strip()
    page_list = pages.split(',')
    ntsd[para] = page_list
    mss[para] = 0

In [ ]:
ntsd['1.2.3.4']

In [ ]:
s2nd = {}

In [ ]:
for line in snd[:-1]:
    page = line.split(" : ")[0].strip()
    sbn_pages_list.append(page)
    paras = line.split(" : ")[1].strip()
    para_list = paras.split(',')
    s2nd[page] = para_list


In [ ]:
s2nd['639']